# Literature Tracker — Exploration

Scratch space for poking at each stage of the pipeline interactively, using the same code the daily/weekly scripts run (`src/lit_pipeline/`). Run `uv sync` once from the project root, then select the `.venv` kernel for this notebook.

Needs a `.env` file in the project root (copy `.env.example`) with `ANTHROPIC_API_KEY` and either `GOOGLE_SERVICE_ACCOUNT_FILE` or `GOOGLE_SERVICE_ACCOUNT_JSON` set, and a real `sheet_id` in `config/settings.yaml`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from anthropic import Anthropic

from lit_pipeline.config import load_settings
from lit_pipeline import sheets_store
from lit_pipeline.arxiv_client import fetch_candidates, download_pdf_bytes
from lit_pipeline.pdf_extract import extract_pdf_text
from lit_pipeline.llm_triage import triage_paper
from lit_pipeline.llm_deep_read import deep_read_paper

settings = load_settings()
client = Anthropic()
settings

## 1. See what arXiv returns for your queries (no writes yet)

In [ ]:
candidates = fetch_candidates(settings.arxiv)
df = pd.DataFrame([c.__dict__ for c in candidates])
df

## 2. Try triage on a single candidate

In [ ]:
sample = candidates[0]
result, usage = triage_paper(client, settings.triage, settings.interests, sample)
print(f"tokens: {usage.input_tokens} in / {usage.output_tokens} out -- ${usage.cost_usd:.5f}")
result

## 3. Try a full deep-read on one paper (costs real Opus tokens)

In [ ]:
pdf_bytes = download_pdf_bytes(sample.pdf_url)
pdf_text = extract_pdf_text(pdf_bytes)
deep_result, deep_usage = deep_read_paper(client, settings.deep_read, settings.interests, sample, pdf_text)
print(f"tokens: {deep_usage.input_tokens} in / {deep_usage.output_tokens} out -- ${deep_usage.cost_usd:.4f}")
deep_result

## 4. Look at what's currently in the Google Sheet

In [ ]:
papers_ws, deep_reads_ws = sheets_store.open_sheets(settings.google_sheets)
papers_df = pd.DataFrame(papers_ws.get_all_records())
papers_df

In [ ]:
deep_reads_df = pd.DataFrame(deep_reads_ws.get_all_records())
deep_reads_df